# الفصل 63: الحوسبة الفعالة والذكاء على الحافة

> طبقة الحدس ليس كل تطبيق يحتاج GPT-4؛ الاتجاه المتسارع في 2024-2025 هو إثبات أن نماذج صغيرة جدًا (1-4 مليار معامل)، إذا دُرِّبت بعناية فائقة على بيانات عالية الجودة (استراتيجية "Textbooks Are All You Need" من...

- **الجزء:** 12 — الاتجاهات المستقبلية
- **الفصل:** 63
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 62: المعماريات التالية لما بعد Transformer
- **الوسوم:** Edge AI، Quantization، BitNet، On-device

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch63-edge-computing.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch63-edge-computing.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس ليس كل تطبيق يحتاج GPT-4؛ الاتجاه المتسارع في 2024-2025 هو إثبات أن نماذج صغيرة جدًا (1-4 مليار معامل)، إذا دُرِّبت بعناية فائقة على بيانات عالية الجودة (استراتيجية "Textbooks Are All You Need" من...

> **بيانات الفصل**
>
> - **الجزء:** 12 — الاتجاهات المستقبلية
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 62: المعماريات التالية لما بعد Transformer


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم فلسفة "صغير لكن قوي" (Small Language Models - SLMs)
- فهم BitNet والتكميم من الجيل التالي: نحو أوزان 1-bit
- فهم الاستدلال على الحافة (Edge Inference)


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**
> ليس كل تطبيق يحتاج GPT-4؛ الاتجاه المتسارع في 2024-2025 هو إثبات أن نماذج صغيرة جدًا (1-4 مليار معامل)، إذا دُرِّبت بعناية فائقة على بيانات عالية الجودة (استراتيجية "Textbooks Are All You Need" من Microsoft Phi)، يمكنها مُجاراة نماذج أكبر منها بعشر مرات في مهام محددة — وتعمل محليًا على هاتف أو حاسوب محمول دون اتصال إنترنت، بخصوصية كاملة وتكلفة استدلال شبه معدومة.


## 2. الخريطة الذهنية


```text
الذكاء على الحافة
├── النموذج
│   ├── SLM (0.5B-4B)
│   ├── بيانات عالية الجودة
│   └── تقطير لمهمة ضيقة
├── الضغط
│   ├── PTQ (INT8/INT4)
│   ├── QAT
│   ├── BitNet (1.58-bit)
│   └── GGUF K-quants
├── التشغيل
│   ├── llama.cpp (CPU/Apple)
│   ├── MLC-LLM (Mobile/WebGPU)
│   ├── Core ML (Apple)
│   └── ExecuTorch
└── القيود
    ├── الذاكرة
    ├── عرض النطاق
    ├── الحرارة/الطاقة
    └── الخصوصية
```


## 3. الشرح المتدرج

### 3.1 فلسفة "صغير لكن قوي" (Small Language Models - SLMs)

> 🟡 **طبقة الآلية**

العامل الحاسم ليس حجم النموذج بل **جودة وكثافة** بيانات التدريب. سلسلة **Phi (Microsoft)** أثبتت أن تدريب نموذج 3.8B معامل على بيانات "تعليمية" مُنتقاة بعناية (مزيج من بيانات مُصنَّعة بجودة كتب مدرسية + بيانات مُصفَّاة بشدة) يُنتج أداءً يُضاهي نماذج 7B-13B على معايير استدلالية.


```text
استراتيجية بيانات Phi (Textbooks Are All You Need):
1. بيانات مُصنَّعة (Synthetic) بجودة عالية:
   - توليد "كتب مدرسية" تركيبية بواسطة GPT-4 تُغطي مفاهيم محددة بعمق
   - توليد تمارين مُحلولة خطوة بخطوة (مشابه لبيانات SFT للاستدلال)
2. تصفية شديدة الصرامة للبيانات الطبيعية (أكثر صرامة من FineWeb-Edu)
   - تصنيف تعليمي (Educational Value Classifier) يستبعد 95%+ من الويب الخام
3. مزيج دقيق (Curriculum): تدرّج من مفاهيم بسيطة لمعقدة أثناء التدريب
```


> ⚖️ **جدول مقارنة: نماذج SLM الرائدة (2024-2025)**

| النموذج | الحجم | الجهة | الميزة الأساسية |
|---|---|---|---|
| Phi-3.5-mini | 3.8B | Microsoft | بيانات تركيبية عالية الكثافة المعرفية |
| Gemma 2 2B/9B | 2B / 9B | Google | Knowledge Distillation من Gemini |
| Llama 3.2 1B/3B | 1B / 3B | Meta | مُحسَّن للحافة (Edge)، دعم متعدد اللغات |
| Qwen2.5 0.5B-3B | 0.5B-3B | Alibaba | تنافسي جدًا في الترميز والرياضيات |
| SmolLM2 | 135M-1.7B | Hugging Face | مفتوح بالكامل (بيانات + كود + أوزان) |

### 3.2 BitNet والتكميم من الجيل التالي: نحو أوزان 1-bit

> 🔴 **طبقة التعمّق**

رأينا في الفصل 56 (ضمنيًا عبر LLMOps) تقنيات التكميم التقليدية (INT8/INT4 - GPTQ/AWQ). **BitNet b1.58 (Microsoft, 2024)** يذهب لأقصى حد ممكن نظريًا: تمثيل كل وزن بثلاث قيم فقط: **{-1, 0, +1}** (أي 1.58 بت فعليًا من ناحية معلوماتية: log₂(3) ≈ 1.58).


In [ ]:
import torch
import torch.nn as nn

class BitLinear(nn.Module):
    """
    تطبيق مفاهيمي لطبقة BitLinear من BitNet b1.58: الأوزان تُكمَّم
    إلى ثلاث قيم فقط {-1, 0, +1} أثناء التمرير الأمامي، لكن تُحفَظ
    نسخة كاملة الدقة (FP16/FP32) للأوزان أثناء التدريب (Straight-Through
    Estimator للتدرجات عبر عملية التكميم غير القابلة للاشتقاق).
    """
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        # الأوزان الكاملة الدقة (تُحدَّث أثناء التدريب فقط - "Latent Weights")
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)

    def quantize_weights_ternary(self, w: torch.Tensor) -> torch.Tensor:
        """
        تكميم الأوزان إلى {-1, 0, +1} باستخدام عتبة = متوسط القيمة المطلقة
        (absmean quantization - الطريقة المستخدمة فعليًا في ورقة BitNet).
        """
        scale = w.abs().mean()  # γ في الورقة الأصلية
        w_scaled = w / (scale + 1e-8)
        # التقريب لأقرب قيمة في {-1, 0, 1} مع القص
        w_quantized = torch.clamp(torch.round(w_scaled), -1, 1)
        return w_quantized, scale

    def quantize_activations_int8(self, x: torch.Tensor) -> torch.Tensor:
        """تكميم التنشيطات إلى 8-bit (absmax quantization لكل صف)."""
        scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
        x_quantized = torch.clamp(
            torch.round(x * 127.0 / scale), -128, 127
        )
        return x_quantized, scale / 127.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Straight-Through Estimator: نستخدم القيم المُكمَّمة في التمرير
        # الأمامي لكن نسمح للتدرج بالمرور كأن التكميم لم يحدث (STE trick)
        w_quant, w_scale = self.quantize_weights_ternary(self.weight)
        w_ste = self.weight + (w_quant - self.weight).detach()  # STE

        x_quant, x_scale = self.quantize_activations_int8(x)
        x_ste = x + (x_quant - x).detach()

        # الضرب المصفوفي يصبح نظريًا بدون عمليات ضرب حقيقية على الهاردوير
        # المُخصَّص (فقط جمع/طرح/تجاهل) - هذا مصدر التوفير الهائل في الطاقة
        output = torch.nn.functional.linear(x_ste, w_ste) * w_scale
        return output


> 🎯 **لماذا هذا ثوري من ناحية الطاقة؟**

الضرب المصفوفي في Transformer التقليدي يتطلب عمليات ضرب-جمع (Multiply-Accumulate) الكاملة الدقة، وهي الأكثر استهلاكًا للطاقة في معالجات GPU/NPU. عندما تكون الأوزان في {-1, 0, +1} فقط، يتحوّل الضرب إلى **عمليات جمع/طرح/تجاهل فقط** — ورقة BitNet تُقدِّر توفيرًا في استهلاك الطاقة يصل إلى **71.4× مقارنة بـ FP16** على هاردوير مُخصَّص (ASIC مستقبلي)، رغم أن المكاسب على GPU الحالية (غير مُحسَّنة لهذا النمط) أقل دراماتيكية حاليًا.

### 3.3 الاستدلال على الحافة (Edge Inference)

> ⚖️ **جدول مقارنة: أطر عمل الاستدلال على الحافة**

| الإطار | المنصة المستهدفة | التقنية الأساسية | أفضل استخدام |
|---|---|---|---|
| llama.cpp (GGUF) | CPU/Apple Silicon/أي جهاز | تكميم K-quants متعدد المستويات | تشغيل محلي عام، لا يحتاج GPU |
| MLC-LLM | هواتف (iOS/Android)، WebGPU | تحويل تلقائي لكود مُحسَّن لكل منصة | تطبيقات موبايل جاهزة |
| Core ML (Apple) | أجهزة Apple (iPhone/Mac) | Neural Engine المُخصَّص | أفضل أداء/طاقة على أجهزة Apple حصرًا |
| ONNX Runtime Mobile | متعدد المنصات | رسم بياني مُحسَّن عام | مرونة عبر منصات متعددة |
| ExecuTorch (PyTorch) | أجهزة محمولة/مُدمَجة | تصدير PyTorch مباشر محسَّن | تكامل سلس مع خط أنابيب PyTorch |


In [ ]:
def estimate_slm_feasibility_on_device(
    model_params_billions: float,
    quantization_bits: int,
    device_ram_gb: float,
    target_tokens_per_second: float,
    device_memory_bandwidth_gbps: float,
) -> dict:
    """
    تقدير جدوى تشغيل نموذج صغير على جهاز حافة محدد، بناءً على قيدين
    جوهريين: الذاكرة المتاحة، وعرض النطاق الترددي للذاكرة (العامل
    الحاكم الفعلي لسرعة الاستدلال في معظم الحالات - وليس قوة المعالج).
    """
    # حجم النموذج التقريبي بعد التكميم
    model_size_gb = (model_params_billions * 1e9 * quantization_bits / 8) / 1e9

    # هامش أمان للذاكرة (KV cache + overhead النظام)
    required_ram_gb = model_size_gb * 1.3

    fits_in_memory = required_ram_gb <= device_ram_gb

    # الاستدلال في نماذج اللغة غالبًا Memory-Bound: السرعة القصوى النظرية
    # = عرض النطاق الترددي للذاكرة / حجم النموذج (لكل رمز يجب قراءة كل الأوزان مرة)
    theoretical_max_tokens_per_sec = (
        device_memory_bandwidth_gbps * 1e9 / (model_size_gb * 1e9)
    )

    meets_speed_target = theoretical_max_tokens_per_sec >= target_tokens_per_second

    return {
        "model_size_gb": round(model_size_gb, 2),
        "required_ram_gb": round(required_ram_gb, 2),
        "fits_in_memory": fits_in_memory,
        "theoretical_max_tokens_per_sec": round(theoretical_max_tokens_per_sec, 1),
        "meets_speed_target": meets_speed_target,
        "recommendation": (
            "✅ قابل للتشغيل بكفاءة على هذا الجهاز"
            if fits_in_memory and meets_speed_target
            else "⚠️ يحتاج تكميم أكثر صرامة أو نموذجًا أصغر"
        ),
    }

# مثال: Phi-3.5-mini (3.8B) مُكمَّم 4-bit على هاتف متوسط
result = estimate_slm_feasibility_on_device(
    model_params_billions=3.8,
    quantization_bits=4,
    device_ram_gb=8.0,
    target_tokens_per_second=15.0,
    device_memory_bandwidth_gbps=25.6,  # نموذجي لهاتف متوسط المدى
)
print(result)
# النتيجة التقريبية: model_size_gb≈1.9, required_ram≈2.47,
# fits_in_memory=True, max_tokens/sec≈13.5 → قد لا يفي بالضبط بالهدف 15


> 💡 **نصيحة عملية**: القاعدة الذهبية لتقدير سرعة الاستدلال على أي جهاز (GPU أو حافة): معظم أعباء توليد النصوص **محدودة بعرض النطاق الترددي للذاكرة (Memory-Bandwidth-Bound)** وليس بقوة الحوسبة الخام (FLOPs-Bound) — لأن توليد كل رمز جديد يتطلب قراءة كل أوزان النموذج مرة واحدة من الذاكرة. لذا فإن تقليل حجم النموذج عبر التكميم يُحسِّن السرعة بشكل شبه خطي مباشر، بخلاف تحسينات الحوسبة الخام التي تأثيرها أقل أهمية لمرحلة فك التشفير (Decoding) تحديدًا (لكنها تبقى حاسمة لمرحلة Prefill - معالجة الـ prompt الأولي).

### 3.4 التقطير إلى نماذج حافة إنتاجية: خط أنابيب كامل


In [ ]:
class EdgeModelDistillationPipeline:
    """
    خط أنابيب كامل لتقطير نموذج ضخم (مُعلِّم) إلى نموذج حافة صغير
    (تلميذ) جاهز للنشر، يجمع بين: Knowledge Distillation (الفصل 25
    من الجزء السادس - Model Reuse) + Quantization-Aware Training +
    تحسين خاص لمهمة ضيقة (Task-Specific Distillation).
    """
    def __init__(self, teacher_model, student_architecture_config: dict,
                 target_device_constraints: dict):
        self.teacher = teacher_model
        self.student_config = student_architecture_config
        self.constraints = target_device_constraints  # RAM، bandwidth، إلخ

    def generate_distillation_dataset(self, task_prompts: list,
                                       n_samples_per_prompt: int = 5) -> list:
        """
        توليد بيانات تقطير مُخصَّصة للمهمة المستهدفة: استجابات عالية
        الجودة من النموذج المُعلِّم تُستخدم كـ "soft targets" أو
        بيانات SFT مباشرة للتلميذ، مُركَّزة فقط على نطاق المهمة
        المستهدفة (لا حاجة لتغطية عامة واسعة كالنموذج الأصلي).
        """
        dataset = []
        for prompt in task_prompts:
            for _ in range(n_samples_per_prompt):
                teacher_response = self.teacher.generate(
                    prompt, temperature=0.7, max_tokens=512
                )
                dataset.append({"prompt": prompt, "response": teacher_response})
        return dataset

    def distill_with_qat(self, student_model, distillation_dataset: list,
                          quantization_bits: int = 4) -> nn.Module:
        """
        تدريب التلميذ مع Quantization-Aware Training (QAT): محاكاة
        تأثير التكميم أثناء التدريب نفسه (وليس بعده - Post-Training
        Quantization فقط)، لتعويض النموذج عن فقدان الدقة قبل حدوثه.
        """
        from torch.quantization import prepare_qat, get_default_qat_qconfig

        student_model.qconfig = get_default_qat_qconfig('fbgemm')
        student_prepared = prepare_qat(student_model, inplace=False)

        optimizer = torch.optim.AdamW(student_prepared.parameters(), lr=5e-5)

        for batch in self._batchify(distillation_dataset, batch_size=8):
            # دمج Hard-label loss (SFT عادي) مع Soft-label loss (KD -
            # راجع الفصل 25 لتفاصيل compute_distillation_loss الكاملة)
            loss = self._compute_combined_loss(student_prepared, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        from torch.quantization import convert
        student_quantized = convert(student_prepared.eval(), inplace=False)
        return student_quantized

    def validate_against_constraints(self, distilled_model) -> dict:
        """التحقق النهائي من مطابقة النموذج المُقطَّر لقيود الجهاز المستهدف."""
        model_size_mb = sum(
            p.numel() * p.element_size() for p in distilled_model.parameters()
        ) / (1024 ** 2)

        passes_ram_constraint = (
            model_size_mb / 1024 <= self.constraints.get("max_ram_gb", 4.0)
        )
        return {
            "model_size_mb": round(model_size_mb, 1),
            "passes_ram_constraint": passes_ram_constraint,
        }

    def _batchify(self, data, batch_size):
        for i in range(0, len(data), batch_size):
            yield data[i:i+batch_size]

    def _compute_combined_loss(self, model, batch):
        raise NotImplementedError("يُطبَّق بحسب واجهة النموذج - راجع الفصل 25")


## 4. مثال محلول خطوة بخطوة

**المعطيات:** نموذج 3.8B، تكميم 4-bit، هاتف بذاكرة 8GB وعرض نطاق 25.6 GB/s.

**حجم النموذج:**

\[
3.8 \times 10^9 \times \frac{4}{8} = 1.9 \text{ GB}
\]

**الذاكرة المطلوبة مع هامش 30%:** \(1.9 \times 1.3 = 2.47\) GB. يناسب جهازًا بذاكرة 8GB.

**السرعة النظرية القصوى (Decode):**

\[
\text{tokens/s} = \frac{25.6}{2.47} \approx 10.4
\]

إذا كان الهدف 15 token/s، فالنموذج لا يحققه نظريًا. الحل: نموذج أصغر أو تكميم 3-bit.

**مقارنة:**

| النموذج | الحجم 4-bit | ذاكرة مطلوبة | سرعة نظرية |
|---|---:|---:|---:|
| 1.5B | 0.75 GB | 0.98 GB | 26.1 tok/s |
| 3.8B | 1.9 GB | 2.47 GB | 10.4 tok/s |
| 7B | 3.5 GB | 4.55 GB | 5.6 tok/s |

**الخلاصة:** على الحافة، عرض النطاق الترددي هو العامل الحاكم، لا عدد المعاملات وحده. اختر أصغر نموذج يحقق الجودة المطلوبة، ثم كمّمه.


## 5. نصائح وحيل

> 💡 **ابدأ من المهمة لا من النموذج**: حدّد دقة مقبولة على مجموعة اختبار حقيقية، ثم اختر أصغر نموذج يحققها.

> 💡 **استخدم QAT إذا كانت الجودة حساسة**: التكميم بعد التدريب قد يُضعف الدقة؛ QAT يعوّض أثناء التدريب.

> 💡 **اختر GGUF Q4_K_M كبداية**: توازن جيد بين الحجم والجودة على CPU. جرّب Q5/Q8 إذا كانت الذاكرة تسمح.

> 💡 **قِس الحرارة والبطارية**: الأداء الأولي قد ينخفض بعد دقائق بسبب thermal throttling. اختبر جلسة 10 دقائق.

> 💡 **اعتبرها فرصة خصوصية**: المعالجة على الجهاز تعني عدم إرسال البيانات للخادم. وثّق ذلك للمستخدم.


## 6. أخطاء شائعة ومصائد

- قياس "قوة" SLM فقط بعدد المعاملات دون مراعاة جودة بيانات التدريب — Phi-3.5 (3.8B) يتفوق على نماذج أقدم أكبر بكثير منه بفضل جودة البيانات فقط.
- نشر BitNet أو تقنيات 1-bit على GPU عام (غير مُخصَّص) متوقعين تسريعًا فوريًا — المكاسب الحقيقية تحتاج هاردوير/kernels مُخصَّصة لم تنتشر تجاريًا بعد على نطاق واسع (2025).
- تجاهل اختبار النموذج المُقطَّر على بيانات **خارج توزيع** بيانات التقطير الضيقة — نموذج حافة مُخصَّص لمهمة ضيقة جدًا قد "ينسى" قدرات عامة كانت موجودة في النموذج المُعلِّم، فتحقق من حدود النطاق المقبولة بوضوح قبل النشر.


## 7. جدول مقارنة

| التقنية | متى تُستخدم | الجودة | التعقيد |
|---|---|---|---|
| PTQ (INT8/INT4) | بعد التدريب، سرعة | جيدة غالبًا | منخفض |
| QAT | جودة حساسة | أفضل | متوسط |
| BitNet 1.58-bit | عتاد مخصص | واعدة | مرتفع/بحثي |
| GGUF Q4_K_M | CPU/Apple | جيدة | منخفض |
| Distillation + QAT | حافة إنتاجية | ممتازة للمهمة | مرتفع |

| الإطار | المنصة | الأفضل لـ |
|---|---|---|
| llama.cpp | CPU/Apple/أي جهاز | تشغيل محلي عام |
| MLC-LLM | iOS/Android/WebGPU | تطبيقات موبايل |
| Core ML | Apple | أفضل أداء على Apple |
| ExecuTorch | محمول/مدمج | تكامل PyTorch |


## 8. ملخص أفضل الممارسات

- ✅ ابدأ دائمًا بسؤال: "هل تطبيقي يحتاج فعلًا نموذجًا ضخمًا، أم أن نموذجًا صغيرًا مُقطَّرًا بعناية على مهمتي الضيقة كافٍ؟" — التوفير في التكلفة والكمون غالبًا يستحق الاستثمار.
- ✅ استخدم `estimate_slm_feasibility_on_device` (أو ما يعادلها) **قبل** الاستثمار في تدريب/تقطير نموذج حافة، لتفادي مفاجآت عدم الملاءمة بعد فوات الأوان.
- ✅ لتطبيقات الخصوصية الحرجة (طبية، قانونية، مؤسسية داخلية)، النشر على الحافة (Edge/On-Device) ليس خيار أداء فقط بل **متطلب امتثال** في كثير من الأحيان.
- ✅ تابع GGUF quantization levels (Q4_K_M، Q5_K_M، Q8_0) لاختيار التوازن الأمثل بين الحجم والجودة حسب الجهاز المستهدف.


## 9. أسئلة مقابلات

**سؤال:** لماذا يُعتبر عرض النطاق الترددي للذاكرة (Memory Bandwidth) غالبًا أهم من قوة المعالج الخام (FLOPs) عند تقييم سرعة استدلال LLM، وكيف يؤثر هذا على استراتيجية التكميم؟

> **إجابة نموذجية:** توليد كل رمز جديد في نموذج لغوي (مرحلة Decoding التسلسلية) يتطلب قراءة **كل** أوزان النموذج من الذاكرة مرة واحدة لكل رمز مُولَّد، لأن الحوسبة الفعلية لكل رمز (ضرب مصفوفي لمتجه واحد) صغيرة جدًا مقارنة بحجم البيانات التي يجب نقلها. هذا يجعل العملية "محدودة بعرض النطاق الترددي" (Memory-Bandwidth-Bound) لا بقوة المعالجة الخام؛ بمعنى أن مضاعفة سرعة المعالج دون تحسين سرعة نقل الذاكرة لن يُسرِّع التوليد كثيرًا. هذا يجعل **التكميم** (تقليل حجم كل وزن من FP16 إلى INT4 مثلًا) الأداة الأكثر تأثيرًا مباشرًا على سرعة الاستدلال، لأنه يُقلِّص حجم البيانات المنقولة من الذاكرة بشكل مباشر ونسبي — تخفيض الدقة من 16-bit إلى 4-bit يُقارب نظريًا تسريعًا بمعامل 4× في مرحلة Decoding تحديدًا (بخلاف مرحلة Prefill الأكثر اعتمادًا على FLOPs الخام).


## 10. تمرين عملي

**الهدف:** تجهيز نموذج عربي صغير للعمل على حاسوب محمول.

**الخطوات:**
1. اختر نموذجًا 1-2B وفهرس بيانات اختبار عربية (100 مثال).
2. قِس الدقة قبل التكميم.
3. صدّر النموذج إلى GGUF Q4_K_M وQ5_K_M.
4. قِس: الحجم، الذاكرة، tokens/s، والدقة بعد التكميم.
5. اختبر جلسة 10 دقائق لرصد الحرارة/التباطؤ.
6. اكتب توصية: أي نسخة تختار ولماذا.

**معيار النجاح:** فقدان دقة أقل من 3%، وسرعة ≥ 10 token/s على الجهاز المستهدف.


## 11. مراجع للتعمّق

- Abdin et al. (2024, Microsoft): *Phi-3 Technical Report: A Highly Capable Language Model Locally on Your Phone*
- Ma et al. (2024, Microsoft): *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits* (BitNet b1.58)
- Gunter et al. (2024, Apple): *Apple Intelligence Foundation Language Models*
- Allal et al. (2024, Hugging Face): *SmolLM2: When Smol Goes Big*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الاتجاهات المستقبلية / الفصل 63: الحوسبة الفعالة والذكاء على الحافة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
